# Financial Sentiment Analysis

This notebook implements sentiment analysis for financial news and social media data using AWS Comprehend and custom models. We'll:
1. Collect financial news data
2. Use AWS Comprehend for sentiment analysis
3. Implement custom FinBERT model for finance-specific sentiment
4. Create a sentiment scoring system
5. Deploy the model to production

In [ ]:
# Import required libraries
import boto3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm
import sagemaker
from sagemaker import get_execution_role

# Initialize AWS clients
comprehend = boto3.client('comprehend')
s3 = boto3.client('s3')
session = sagemaker.Session()
role = get_execution_role()

# Initialize FinBERT model
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

## AWS Comprehend Sentiment Analysis

First, let's implement sentiment analysis using AWS Comprehend for general sentiment detection.

In [ ]:
# Function to analyze sentiment using AWS Comprehend
def analyze_sentiment_comprehend(text):
    try:
        response = comprehend.detect_sentiment(
            Text=text,
            LanguageCode='en'
        )
        return {
            'sentiment': response['Sentiment'],
            'scores': response['SentimentScore']
        }
    except Exception as e:
        print(f"Error analyzing sentiment: {str(e)}")
        return None

# Sample financial news texts
sample_texts = [
    "The company reported strong earnings, beating market expectations.",
    "Market uncertainty leads to increased volatility in tech stocks.",
    "Economic indicators suggest potential recession risks ahead.",
    "New policy changes boost investor confidence in emerging markets."
]

# Analyze sentiment for sample texts
for text in sample_texts:
    sentiment = analyze_sentiment_comprehend(text)
    print(f"\nText: {text}")
    print(f"Sentiment: {sentiment['sentiment']}")
    print(f"Scores: {json.dumps(sentiment['scores'], indent=2)}")

## Custom FinBERT Implementation

Now, let's use the FinBERT model for finance-specific sentiment analysis.

In [ ]:
# Function to analyze sentiment using FinBERT
def analyze_sentiment_finbert(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    outputs = model(**inputs)
    predictions = F.softmax(outputs.logits, dim=-1)
    
    labels = ['negative', 'neutral', 'positive']
    scores = {label: score.item() for label, score in zip(labels, predictions[0])}
    sentiment = labels[predictions.argmax().item()]
    
    return {
        'sentiment': sentiment,
        'scores': scores
    }

# Analyze the same texts with FinBERT
print("FinBERT Analysis:")
for text in sample_texts:
    sentiment = analyze_sentiment_finbert(text)
    print(f"\nText: {text}")
    print(f"Sentiment: {sentiment['sentiment']}")
    print(f"Scores: {json.dumps(sentiment['scores'], indent=2)}")

# Compare results between Comprehend and FinBERT
def compare_sentiments(text):
    comprehend_result = analyze_sentiment_comprehend(text)
    finbert_result = analyze_sentiment_finbert(text)
    
    return pd.DataFrame({
        'Model': ['AWS Comprehend', 'FinBERT'],
        'Sentiment': [comprehend_result['sentiment'], finbert_result['sentiment']],
        'Confidence': [max(comprehend_result['scores'].values()), max(finbert_result['scores'].values())]
    })

print("\nComparison of Models:")
for text in sample_texts:
    print(f"\nAnalyzing: {text}")
    print(compare_sentiments(text))

## Model Deployment

Let's package and deploy our sentiment analysis models to SageMaker endpoints.

In [ ]:
# Save FinBERT model for deployment
model_path = 'models/finbert'
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

# Upload model to S3
model_artifact = session.upload_data(
    model_path,
    bucket=session.default_bucket(),
    key_prefix='models/finbert'
)

# Create deployment script
%%writefile code/sentiment_inference.py
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

def model_fn(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    return model, tokenizer

def predict_fn(data, model_and_tokenizer):
    model, tokenizer = model_and_tokenizer
    
    # Process input
    inputs = tokenizer(data['text'], return_tensors="pt", padding=True, truncation=True)
    
    # Get prediction
    with torch.no_grad():
        outputs = model(**inputs)
    predictions = F.softmax(outputs.logits, dim=-1)
    
    # Format output
    labels = ['negative', 'neutral', 'positive']
    scores = {label: score.item() for label, score in zip(labels, predictions[0])}
    sentiment = labels[predictions.argmax().item()]
    
    return {
        'sentiment': sentiment,
        'scores': scores
    }

# Deploy model
from sagemaker.huggingface import HuggingFaceModel

huggingface_model = HuggingFaceModel(
    model_data=model_artifact,
    role=role,
    transformers_version='4.26.0',
    pytorch_version='1.13.1',
    py_version='py39',
)

predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.xlarge',
    endpoint_name='econosense-sentiment'
)

print("Model deployed successfully to endpoint: econosense-sentiment")